# Phase 1: Data collection for the JUST CIS Course-Advising Assistant

## Step 1: Install libraries

In [1]:
!pip install -q requests beautifulsoup4 pandas lxml

## Step 2: Write the data collection script

In [2]:
%%writefile scrape_plan.py
import os
import re
from datetime import date
import pandas as pd
import requests
from bs4 import BeautifulSoup
PLAN_NO = 766
URL = ("https://www.just.edu.jo/FacultiesandDepartments/it/Departments/cis/"
       f"Pages/viewplan.aspx?planno={PLAN_NO}")
os.makedirs("data/raw", exist_ok=True)
resp = requests.get(URL, timeout=60, headers={"User-Agent": "Mozilla/5.0 (course project)"})
resp.raise_for_status()
html = resp.text
with open(f"data/raw/plan_{PLAN_NO}.html", "w", encoding="utf-8") as f:
    f.write(html)
print(f"Saved page ({len(html):,} characters) on {date.today()}")
soup = BeautifulSoup(html, "lxml")
links = []
for a in soup.find_all("a", href=True):
    text = a.get_text(strip=True)
    if re.search(r"syllab|description|evaluation", text, re.I):
        links.append({"text": text, "href": requests.compat.urljoin(URL, a["href"])})
pd.DataFrame(links).to_csv("data/raw/links.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(links)} syllabus/description links")
from bs4 import Tag
header_re = re.compile(r"\s*(.+?Req\.?)\s*\((\d+)\)\s*Hours?\s*:?\s*", re.I)
rows, section, section_hours = [], None, None
for el in soup.body.descendants:
    if not isinstance(el, Tag):
        continue
    if el.name == "tr":
        cells = [c.get_text(" ", strip=True) for c in el.find_all(["td", "th"], recursive=False)]
        if len(cells) >= 4 and re.fullmatch(r"\d{5,8}", cells[0] or ""):
            rows.append({
                "line_no": cells[0], "code": cells[1], "course": cells[2], "hours": cells[3],
                "prerequisite_raw": cells[4] if len(cells) > 4 else "",
                "teaching_method": cells[5] if len(cells) > 5 else "",
                "section": section, "section_hours": section_hours,
            })
    elif el.find("tr") is None:
        text = el.get_text(" ", strip=True)
        m = header_re.fullmatch(text) if len(text) < 80 else None
        if m:
            section, section_hours = m.group(1).strip(), int(m.group(2))
raw = pd.DataFrame(rows)
raw.to_csv("data/raw/plan_rows_raw.csv", index=False, encoding="utf-8-sig")
RELATIONS = [("prerequisite / pass", "prereq_pass"), ("prerequisite / study", "prereq_study"),
             ("pre./con.", "prereq_or_concurrent"), ("concurrent", "concurrent"),
             ("equivalent", "equivalents")]
REL_COLS = [c for _, c in RELATIONS]
def norm_rel(text):
    t = re.sub(r"\s+", " ", text.strip().lower())
    for key, col in RELATIONS:
        if t == key or t.replace(" ", "") == key.replace(" ", ""):
            return col
    return None
def group_codes(items):
    groups = {}
    for code, name in items:
        groups.setdefault(re.sub(r"\s+", " ", name.strip().lower()), []).append(code)
    return "; ".join("/".join(dict.fromkeys(codes)) for codes in groups.values())
courses, current, unknown = [], None, []
for r in rows:
    rel = r["prerequisite_raw"].strip()
    if rel in ("", "---"):
        current = {**r, **{c: [] for c in REL_COLS}}
        courses.append(current)
    elif current is not None:
        col = norm_rel(rel)
        if col:
            current[col].append((r["code"], r["course"]))
        else:
            unknown.append(r)
courses = pd.DataFrame(courses)
for col in REL_COLS:
    courses[col] = courses[col].apply(group_codes)
courses.to_csv("data/plan_courses.csv", index=False, encoding="utf-8-sig")
print(f"Parsed {len(raw)} raw rows -> {len(courses)} main courses in data/plan_courses.csv")
if unknown:
    print(f"{len(unknown)} rows with unrecognised relation text (check by hand):")
    for r in unknown:
        print("  ", r["code"], "|", r["prerequisite_raw"])
if not courses.empty:
    courses["hours_num"] = pd.to_numeric(courses["hours"], errors="coerce")
    print("\nSections found and their stated hours:")
    print(courses.groupby(["section", "section_hours"]).size().rename("course_rows"))
    print("\nRows with unreadable hours:", courses["hours_num"].isna().sum())
print("\nNEXT: open the page in a browser and compare 10 random rows by hand.")

Writing scrape_plan.py


## Step 3: Download the plan and build the course table

In [3]:
!python scrape_plan.py

Saved page (427,643 characters) on 2026-10-30
Saved 768 syllabus/description links
Parsed 256 raw rows -> 115 main courses in data/plan_courses.csv

Sections found and their stated hours:
section                     section_hours
Department Compulsory Req.  71               26
Department Elective Req.    12               21
Faculty Compulsory Req.     24               10
University Compulsory Req.  16                6
University Elective Req.    9                52
Name: course_rows, dtype: int64

Rows with unreadable hours: 0

NEXT: open the page in a browser and compare 10 random rows by hand.


## Step 4: Look at the course table

In [4]:
import os
import pandas as pd
courses = pd.read_csv("data/plan_courses.csv", dtype=str)
print(len(courses), "main courses")
cols = ["code", "course", "hours", "section", "section_hours",
        "prereq_pass", "prereq_study", "prereq_or_concurrent", "concurrent", "equivalents"]
pd.set_option("display.max_rows", 200)
courses[cols]

115 main courses


,code,course,hours,section,section_hours,prereq_pass,prereq_study,prereq_or_concurrent,concurrent,equivalents
0,ARB102,communication skills in arabic,3,University Compulsory Req.,16,NaN,NaN,NaN,NaN,NaN
1,HSS110,leader and social responsibility,3,University Compulsory Req.,16,NaN,NaN,NaN,NaN,HSS110
2,HSS119,entrepreneurship and innovation,2,University Compulsory Req.,16,NaN,NaN,NaN,NaN,HSS119
3,MS100,military sciences,3,University Compulsory Req.,16,NaN,NaN,NaN,NaN,NaN
4,LG101,communication skills in english,3,University Compulsory Req.,16,ENG099/LG099,NaN,NaN,NaN,LG112
5,LG103,life skills,2,University Compulsory Req.,16,NaN,NaN,NaN,NaN,HSS129
6,ARB200,appreciation of literary texts,3,University Elective Req.,9,NaN,NaN,NaN,NaN,NaN
7,HSS100NUR,"health promotion (for non medicine ,nursing an...",3,University Elective Req.,9,NaN,NaN,NaN,NaN,NUR100
8,HSS100O.T,disability and the society(not allowed for reh...,3,University Elective Req.,9,NaN,NaN,NaN,NaN,O.T100
9,HSS100P.T,wellness and life styles (not for physical & o...,3,University Elective Req.,9,NaN,NaN,NaN,NaN,P.T100


## Step 5: Automatic checks

In [5]:
raw = pd.read_csv("data/raw/plan_rows_raw.csv", dtype=str)
known_codes = set(raw["code"])
print("Courses per section (with stated section hours):")
print(courses.groupby(["section", "section_hours"]).size().rename("main_courses"), "\n")
print("Rows with non-numeric hours:", pd.to_numeric(courses["hours"], errors="coerce").isna().sum())
print("Rows with no section:", courses["section"].isna().sum())
problems = []
for _, r in courses.iterrows():
    for col in ["prereq_pass", "prereq_study", "prereq_or_concurrent", "concurrent"]:
        if pd.isna(r[col]):
            continue
        for group in r[col].split("; "):
            for code in group.split("/"):
                if code == r["code"]:
                    problems.append(f"{r['code']} lists itself in {col}")
                elif code not in known_codes:
                    problems.append(f"{r['code']}: {code} in {col} not found on the page")
print("Prerequisite problems:", problems if problems else "none")
dups = courses["code"][courses["code"].duplicated()].tolist()
print("Duplicate course codes:", dups if dups else "none")

Courses per section (with stated section hours):
section                     section_hours
Department Compulsory Req.  71               26
Department Elective Req.    12               21
Faculty Compulsory Req.     24               10
University Compulsory Req.  16                6
University Elective Req.    9                52
Name: main_courses, dtype: int64 

Rows with non-numeric hours: 0
Rows with no section: 0
Prerequisite problems: none
Duplicate course codes: none


## Step 6: Save a random 10-row sample for the manual check

In [6]:
check = courses.sample(min(10, len(courses)), random_state=42)[cols]
check.to_csv("data/manual_check_10_rows.csv", index=False, encoding="utf-8-sig")
check

,code,course,hours,section,section_hours,prereq_pass,prereq_study,prereq_or_concurrent,concurrent,equivalents
81,CIs381,mobile application development,2,Department Compulsory Req.,71,NaN,HSS112SE; CIs341,NaN,NaN,HI335
4,LG101,communication skills in english,3,University Compulsory Req.,16,ENG099/LG099,NaN,NaN,NaN,LG112
40,HSS202PP,natural plants of jordan (for non agriculture ...,3,University Elective Req.,9,NaN,NaN,NaN,NaN,PP202
69,MATH140,elements of linear algebra,3,Department Compulsory Req.,71,NaN,NaN,NaN,NaN,NaN
10,HSS100ADS,oral & dental health,3,University Elective Req.,9,NaN,NaN,NaN,NaN,ADS100
45,HSS213,individual and society,3,University Elective Req.,9,NaN,NaN,NaN,NaN,NaN
70,CS284,analysis and design of algorithms,3,Department Compulsory Req.,71,NaN,HSS211CS/CS211,NaN,NaN,NaN
66,HSS241MATH,discrete mathematics,3,Faculty Compulsory Req.,24,NaN,NaN,NaN,NaN,MATH241
47,HSS214VM,animal products and public health(for non vet...,3,University Elective Req.,9,NaN,NaN,NaN,NaN,VM214
11,HSS101AI,ai applications,3,University Elective Req.,9,NaN,NaN,NaN,NaN,NaN


## Step 7: Make the 5-row data sample for the repository

In [7]:
os.makedirs("data/sample", exist_ok=True)
courses.sample(min(5, len(courses)), random_state=1).to_csv(
    "data/sample/plan_sample_5_rows.csv", index=False, encoding="utf-8-sig")
pd.read_csv("data/sample/plan_sample_5_rows.csv", dtype=str)

,line_no,code,course,hours,prerequisite_raw,teaching_method,section,section_hours,prereq_pass,prereq_study,prereq_or_concurrent,concurrent,equivalents
0,1744011,CIs401,multimedia for web design,3,NaN,Blended,Department Elective Req.,12,NaN,HSS211CS/CS211; CIs201,NaN,NaN,NaN
1,822121,HSS212VM,pet animal care,3,NaN,Electronic Course,University Elective Req.,9,NaN,NaN,NaN,NaN,VM212
2,2511080,LG108,turkish language,3,---,Blended,University Elective Req.,9,NaN,NaN,NaN,NaN,NaN
3,1744210,CIs421,database applications,3,NaN,On Campus,Department Compulsory Req.,71,NaN,HSS221CIS,NaN,NaN,HI421
4,901400,MATH140,elements of linear algebra,3,---,Blended,Department Compulsory Req.,71,NaN,NaN,NaN,NaN,NaN


## Step 8: Numbers for the data card

In [8]:
from datetime import date
try:
    n_links = len(pd.read_csv("data/raw/links.csv"))
except pd.errors.EmptyDataError:
    n_links = 0
print("Collected on:", date.today())
print("Raw rows on the page:", len(raw))
print("Main courses:", len(courses))
print("Sections:", courses["section"].nunique())
print("Syllabus / description links:", n_links)

Collected on: 2026-10-30
Raw rows on the page: 256
Main courses: 115
Sections: 5
Syllabus / description links: 768


## Step 9: Download everything

In [9]:
!zip -qr phase1_data.zip data scrape_plan.py
from google.colab import files
files.download("phase1_data.zip")